# LottoBench Community Benchmark Report

## Goal
Review a frozen, synthetic, forward-only benchmark without making claims about operated lotteries or future returns.

## Setup
The notebook uses only the attached Kaggle dataset. Locally it falls back to the sibling dataset bundle.

In [ ]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

kaggle_data = Path('/kaggle/input/lottobench-community-benchmark')
data_dir = kaggle_data if kaggle_data.exists() else Path('../dataset')
manifest = json.loads((data_dir / 'benchmark_manifest.json').read_text())
history_path = data_dir / 'synthetic_history.csv'
results = pd.read_csv(data_dir / 'benchmark_results.csv')
history = pd.read_csv(history_path)
print(f"Using {data_dir}; benchmark {manifest['benchmark_version']}")

## Checks

In [ ]:
actual_digest = hashlib.sha256(history_path.read_bytes()).hexdigest()
assert actual_digest == manifest['dataset_sha256']
assert results['dataset_sha256'].nunique() == 1
assert results['dataset_sha256'].iloc[0] == actual_digest
assert (results['budget'] == manifest['evaluation']['budget']).all()
assert (results['holdout'] == manifest['evaluation']['holdout']).all()
print(f"Validated {len(history)} synthetic draws and {len(results)} result rows.")

## Results

In [ ]:
columns = ['provider', 'pair_coverage', 'number_coverage', 'mean_jaccard_diversity', 'unpopularity_lift', 'expected_roi_per_ticket', 'hit_recall']
ranked = results[columns].sort_values('pair_coverage', ascending=False).reset_index(drop=True)
ranked

In [ ]:
ax = ranked.sort_values('pair_coverage').plot.barh(x='provider', y='pair_coverage', legend=False, figsize=(8, 4), color='#2878b5')
ax.set(title='Synthetic forward-holdout pair coverage', xlabel='Mean pair coverage', ylabel='')
plt.tight_layout()
plt.show()

## Takeaways

The table ranks implementation behavior on the frozen synthetic snapshot. Pair coverage is the primary metric. Expected ROI is a modeled diagnostic and remains negative. Hit recall sits on the fair-draw null of main_k/main_n for every provider. The synthetic history is uniform by construction, so no method can show predictive lift on it; these rows calibrate the harness rather than test any method. Platform publication is not independent validation, endorsement, or gambling advice.